In [4]:
import pandas as pd
import numpy as np
import os
from pathlib import Path


In [5]:
# ------------------------------------------------------------
# 1. CONFIGURACIÓN DE RUTAS
# ------------------------------------------------------------
# Rutas de salida dentro de Google Colab
PROCESSED_PATH = Path("/content/data/processed")
STAR_SCHEMA_PATH = Path("/content/data/star_schema")

PROCESSED_PATH.mkdir(parents=True, exist_ok=True)
STAR_SCHEMA_PATH.mkdir(parents=True, exist_ok=True)

# URL raw del CSV procesado en GitHub
GITHUB_CSV_URL = "https://raw.githubusercontent.com/galvanjuanmanuel17-BIT/data-warehouse-sales-project/main/data/processed/online_retail_sample.csv"

# Carga del dataset procesado
df = pd.read_csv(GITHUB_CSV_URL)

print("Dataset cargado correctamente desde GitHub")
print("Filas iniciales:", df.shape[0])
print("Columnas iniciales:", df.shape[1])

df.head()

Dataset cargado correctamente desde GitHub
Filas iniciales: 50000
Columnas iniciales: 10


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Region,Categoria
0,555200,71459,HANGING JAM JAR T-LIGHT HOLDER,24,2011-06-01 12:05:00,0.85,17315.0,United Kingdom,Europe,Iluminación
1,554974,21128,GOLD FISHING GNOME,4,2011-05-27 17:14:00,6.95,14031.0,United Kingdom,Europe,Otros
2,550972,21086,SET/6 RED SPOTTY PAPER CUPS,4,2011-04-21 17:05:00,0.65,14031.0,United Kingdom,Europe,Cocina y vajilla
3,576652,22812,PACK 3 BOXES CHRISTMAS PANETTONE,3,2011-11-16 10:39:00,1.95,17198.0,United Kingdom,Europe,Navidad y temporada
4,546157,22180,RETROSPOT LAMP,2,2011-03-10 08:40:00,9.95,13502.0,United Kingdom,Europe,Iluminación


In [6]:
print(df.columns.tolist())

['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country', 'Region', 'Categoria']


In [7]:
# ============================================================
# 2. ESTANDARIZACIÓN DE NOMBRES DE COLUMNAS
# ============================================================

df = df.rename(columns={
    "InvoiceNo": "Invoice_ID",
    "StockCode": "Product_ID",
    "CustomerID": "Customer_ID",
    "Description": "Description",
    "Quantity": "Quantity",
    "InvoiceDate": "InvoiceDate",
    "UnitPrice": "UnitPrice",
    "Country": "Country",
    "Region": "Region",
    "Categoria": "Categoria"
})

print("Columnas luego de renombrar:")
print(df.columns.tolist())

df.head()

Columnas luego de renombrar:
['Invoice_ID', 'Product_ID', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'Customer_ID', 'Country', 'Region', 'Categoria']


,Invoice_ID,Product_ID,Description,Quantity,InvoiceDate,UnitPrice,Customer_ID,Country,Region,Categoria
0,555200,71459,HANGING JAM JAR T-LIGHT HOLDER,24,2011-06-01 12:05:00,0.85,17315.0,United Kingdom,Europe,Iluminación
1,554974,21128,GOLD FISHING GNOME,4,2011-05-27 17:14:00,6.95,14031.0,United Kingdom,Europe,Otros
2,550972,21086,SET/6 RED SPOTTY PAPER CUPS,4,2011-04-21 17:05:00,0.65,14031.0,United Kingdom,Europe,Cocina y vajilla
3,576652,22812,PACK 3 BOXES CHRISTMAS PANETTONE,3,2011-11-16 10:39:00,1.95,17198.0,United Kingdom,Europe,Navidad y temporada
4,546157,22180,RETROSPOT LAMP,2,2011-03-10 08:40:00,9.95,13502.0,United Kingdom,Europe,Iluminación


In [8]:
# ============================================================
# 3. LIMPIEZA MÍNIMA Y FEATURE ENGINEERING
# ============================================================

# Copia de trabajo
df_model = df.copy()

# Tipos de datos
df_model["Invoice_ID"] = df_model["Invoice_ID"].astype(str)
df_model["Product_ID"] = df_model["Product_ID"].astype(str)
df_model["Description"] = df_model["Description"].astype(str).str.strip()
df_model["Country"] = df_model["Country"].astype(str).str.strip()
df_model["Region"] = df_model["Region"].astype(str).str.strip()
df_model["Categoria"] = df_model["Categoria"].astype(str).str.strip()

df_model["InvoiceDate"] = pd.to_datetime(df_model["InvoiceDate"], errors="coerce")
df_model["Quantity"] = pd.to_numeric(df_model["Quantity"], errors="coerce")
df_model["UnitPrice"] = pd.to_numeric(df_model["UnitPrice"], errors="coerce")
df_model["Customer_ID"] = pd.to_numeric(df_model["Customer_ID"], errors="coerce")

# Eliminar nulos críticos
df_model = df_model.dropna(subset=[
    "Invoice_ID",
    "Product_ID",
    "Description",
    "Quantity",
    "InvoiceDate",
    "UnitPrice",
    "Customer_ID",
    "Country",
    "Region",
    "Categoria"
])

# Eliminar devoluciones o cancelaciones si existieran
df_model = df_model[~df_model["Invoice_ID"].str.startswith("C", na=False)]
df_model = df_model[df_model["Quantity"] > 0]
df_model = df_model[df_model["UnitPrice"] > 0]

# Customer_ID como texto, para evitar decimales tipo 17315.0
df_model["Customer_ID"] = df_model["Customer_ID"].astype(int).astype(str)

# Crear Revenue
df_model["Revenue"] = df_model["Quantity"] * df_model["UnitPrice"]

# Crear columnas temporales
df_model["Year"] = df_model["InvoiceDate"].dt.year
df_model["Month"] = df_model["InvoiceDate"].dt.month
df_model["Quarter"] = "Q" + df_model["InvoiceDate"].dt.quarter.astype(str)

print("Filas luego de limpieza:", df_model.shape[0])
print("Columnas luego de limpieza:", df_model.shape[1])
print("Revenue total:", round(df_model["Revenue"].sum(), 2))

df_model.head()

Filas luego de limpieza: 36553
Columnas luego de limpieza: 14
Revenue total: 851215.63


,Invoice_ID,Product_ID,Description,Quantity,InvoiceDate,UnitPrice,Customer_ID,Country,Region,Categoria,Revenue,Year,Month,Quarter
0,555200,71459,HANGING JAM JAR T-LIGHT HOLDER,24,2011-06-01 12:05:00,0.85,17315,United Kingdom,Europe,Iluminación,20.40,2011,6,Q2
1,554974,21128,GOLD FISHING GNOME,4,2011-05-27 17:14:00,6.95,14031,United Kingdom,Europe,Otros,27.80,2011,5,Q2
2,550972,21086,SET/6 RED SPOTTY PAPER CUPS,4,2011-04-21 17:05:00,0.65,14031,United Kingdom,Europe,Cocina y vajilla,2.60,2011,4,Q2
3,576652,22812,PACK 3 BOXES CHRISTMAS PANETTONE,3,2011-11-16 10:39:00,1.95,17198,United Kingdom,Europe,Navidad y temporada,5.85,2011,11,Q4
4,546157,22180,RETROSPOT LAMP,2,2011-03-10 08:40:00,9.95,13502,United Kingdom,Europe,Iluminación,19.90,2011,3,Q1


In [9]:
# ============================================================
# 4. VALIDACIONES DEL DATASET PROCESADO
# ============================================================

print("Nulos por columna:")
print(df_model.isna().sum())

print("\nValidaciones principales:")
print("Quantity <= 0:", (df_model["Quantity"] <= 0).sum())
print("UnitPrice <= 0:", (df_model["UnitPrice"] <= 0).sum())
print("Revenue <= 0:", (df_model["Revenue"] <= 0).sum())
print("Customer_ID nulos:", df_model["Customer_ID"].isna().sum())

print("\nValores únicos:")
print("Facturas únicas:", df_model["Invoice_ID"].nunique())
print("Productos únicos:", df_model["Product_ID"].nunique())
print("Clientes únicos:", df_model["Customer_ID"].nunique())
print("Países únicos:", df_model["Country"].nunique())
print("Regiones únicas:", df_model["Region"].nunique())
print("Categorías únicas:", df_model["Categoria"].nunique())

Nulos por columna:
Invoice_ID     0
Product_ID     0
Description    0
Quantity       0
InvoiceDate    0
UnitPrice      0
Customer_ID    0
Country        0
Region         0
Categoria      0
Revenue        0
Year           0
Month          0
Quarter        0
dtype: int64

Validaciones principales:
Quantity <= 0: 0
UnitPrice <= 0: 0
Revenue <= 0: 0
Customer_ID nulos: 0

Valores únicos:
Facturas únicas: 12514
Productos únicos: 2964
Clientes únicos: 3741
Países únicos: 36
Regiones únicas: 6
Categorías únicas: 7


In [10]:
# ============================================================
# 5. DIM_PRODUCT
# ============================================================

dim_product = (
    df_model
    .sort_values("InvoiceDate")
    .groupby("Product_ID", as_index=False)
    .agg({
        "Description": "last",
        "Categoria": "last"
    })
)

dim_product = dim_product.rename(columns={
    "Description": "Product_Description",
    "Categoria": "Category"
})

dim_product.insert(0, "Product_Key", range(1, len(dim_product) + 1))

print("Dim Product:", dim_product.shape)
dim_product.head()

Dim Product: (2964, 4)


,Product_Key,Product_ID,Product_Description,Category
0,1,10002,INFLATABLE POLITICAL GLOBE,Otros
1,2,10080,GROOVY CACTUS INFLATABLE,Otros
2,3,10120,DOGGY RUBBER,Otros
3,4,10125,MINI FUNKY DESIGN TAPES,Otros
4,5,10133,COLOURING PENCILS BROWN TUBE,Otros


In [11]:
# ============================================================
# 6. DIM_CUSTOMER
# ============================================================

dim_customer = (
    df_model
    .sort_values("InvoiceDate")
    .groupby("Customer_ID", as_index=False)
    .agg({
        "Country": "last",
        "Region": "last"
    })
)

dim_customer.insert(0, "Customer_Key", range(1, len(dim_customer) + 1))

print("Dim Customer:", dim_customer.shape)
dim_customer.head()

Dim Customer: (3741, 4)


,Customer_Key,Customer_ID,Country,Region
0,1,12346,United Kingdom,Europe
1,2,12347,Iceland,Other
2,3,12348,Finland,Europe
3,4,12349,Italy,Europe
4,5,12350,Norway,Europe


In [12]:
# ============================================================
# 7. DIM_TIME
# ============================================================

min_date = df_model["InvoiceDate"].min().date()
max_date = df_model["InvoiceDate"].max().date()

dim_time = pd.DataFrame({
    "Date": pd.date_range(start=min_date, end=max_date, freq="D")
})

dim_time["Date_Key"] = dim_time["Date"].dt.strftime("%Y%m%d").astype(int)
dim_time["Year"] = dim_time["Date"].dt.year
dim_time["Quarter"] = "Q" + dim_time["Date"].dt.quarter.astype(str)
dim_time["Month"] = dim_time["Date"].dt.month
dim_time["Month_Name"] = dim_time["Date"].dt.month_name()
dim_time["Day"] = dim_time["Date"].dt.day
dim_time["Weekday"] = dim_time["Date"].dt.day_name()

dim_time = dim_time[
    [
        "Date_Key",
        "Date",
        "Year",
        "Quarter",
        "Month",
        "Month_Name",
        "Day",
        "Weekday"
    ]
]

print("Dim Time:", dim_time.shape)
dim_time.head()

Dim Time: (374, 8)


,Date_Key,Date,Year,Quarter,Month,Month_Name,Day,Weekday
0,20101201,2010-12-01,2010,Q4,12,December,1,Wednesday
1,20101202,2010-12-02,2010,Q4,12,December,2,Thursday
2,20101203,2010-12-03,2010,Q4,12,December,3,Friday
3,20101204,2010-12-04,2010,Q4,12,December,4,Saturday
4,20101205,2010-12-05,2010,Q4,12,December,5,Sunday


In [13]:
# ============================================================
# 8. FACT_SALES
# ============================================================

fact_sales = df_model.copy()

# Crear fecha sin hora para relacionarla con dim_time
fact_sales["Date"] = fact_sales["InvoiceDate"].dt.normalize()
dim_time["Date"] = pd.to_datetime(dim_time["Date"])

# Relación con dim_product
fact_sales = fact_sales.merge(
    dim_product[["Product_Key", "Product_ID"]],
    on="Product_ID",
    how="left"
)

# Relación con dim_customer
fact_sales = fact_sales.merge(
    dim_customer[["Customer_Key", "Customer_ID"]],
    on="Customer_ID",
    how="left"
)

# Relación con dim_time
fact_sales = fact_sales.merge(
    dim_time[["Date_Key", "Date"]],
    on="Date",
    how="left"
)

# Seleccionar columnas finales de la fact table
fact_sales = fact_sales[
    [
        "Invoice_ID",
        "Product_Key",
        "Customer_Key",
        "Date_Key",
        "Quantity",
        "UnitPrice",
        "Revenue"
    ]
]

# Crear clave técnica de la tabla de hechos
fact_sales.insert(0, "Sales_Line_ID", range(1, len(fact_sales) + 1))

print("Fact Sales:", fact_sales.shape)
fact_sales.head()

Fact Sales: (36553, 8)


,Sales_Line_ID,Invoice_ID,Product_Key,Customer_Key,Date_Key,Quantity,UnitPrice,Revenue
0,1,555200,2367,3144,20110601,24,0.85,20.40
1,2,554974,291,1082,20110527,4,6.95,27.80
2,3,550972,264,1082,20110421,4,0.65,2.60
3,4,576652,1505,3072,20111116,3,1.95,5.85
4,5,546157,927,751,20110310,2,9.95,19.90


In [14]:
# ============================================================
# 9. VALIDACIONES DEL STAR SCHEMA
# ============================================================

print("Validación de claves nulas en fact_sales:")
print("Product_Key nulos:", fact_sales["Product_Key"].isna().sum())
print("Customer_Key nulos:", fact_sales["Customer_Key"].isna().sum())
print("Date_Key nulos:", fact_sales["Date_Key"].isna().sum())

print("\nDuplicados en dimensiones:")
print("Duplicados Product_ID:", dim_product["Product_ID"].duplicated().sum())
print("Duplicados Customer_ID:", dim_customer["Customer_ID"].duplicated().sum())
print("Duplicados Date_Key:", dim_time["Date_Key"].duplicated().sum())

print("\nValidación de cantidad de filas:")
print("Filas dataset procesado:", len(df_model))
print("Filas fact_sales:", len(fact_sales))

print("\nValidación de Revenue:")
print("Revenue dataset procesado:", round(df_model["Revenue"].sum(), 2))
print("Revenue fact_sales:", round(fact_sales["Revenue"].sum(), 2))
print("Diferencia:", round(df_model["Revenue"].sum() - fact_sales["Revenue"].sum(), 2))

Validación de claves nulas en fact_sales:
Product_Key nulos: 0
Customer_Key nulos: 0
Date_Key nulos: 0

Duplicados en dimensiones:
Duplicados Product_ID: 0
Duplicados Customer_ID: 0
Duplicados Date_Key: 0

Validación de cantidad de filas:
Filas dataset procesado: 36553
Filas fact_sales: 36553

Validación de Revenue:
Revenue dataset procesado: 851215.63
Revenue fact_sales: 851215.63
Diferencia: 0.0


In [15]:
# ============================================================
# 10. EXPORTACIÓN DE TABLAS STAR SCHEMA
# ============================================================

fact_sales.to_csv(STAR_SCHEMA_PATH / "fact_sales.csv", index=False)
dim_product.to_csv(STAR_SCHEMA_PATH / "dim_product.csv", index=False)
dim_customer.to_csv(STAR_SCHEMA_PATH / "dim_customer.csv", index=False)
dim_time.to_csv(STAR_SCHEMA_PATH / "dim_time.csv", index=False)

print("Tablas exportadas correctamente:")
print(STAR_SCHEMA_PATH / "fact_sales.csv")
print(STAR_SCHEMA_PATH / "dim_product.csv")
print(STAR_SCHEMA_PATH / "dim_customer.csv")
print(STAR_SCHEMA_PATH / "dim_time.csv")

Tablas exportadas correctamente:
/content/data/star_schema/fact_sales.csv
/content/data/star_schema/dim_product.csv
/content/data/star_schema/dim_customer.csv
/content/data/star_schema/dim_time.csv


In [16]:
# ============================================================
# 11. DESCARGA DE ARCHIVOS DESDE GOOGLE COLAB
# ============================================================

from google.colab import files

files.download("/content/data/star_schema/fact_sales.csv")
files.download("/content/data/star_schema/dim_product.csv")
files.download("/content/data/star_schema/dim_customer.csv")
files.download("/content/data/star_schema/dim_time.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>